# 실험 03: 다양한 실전 Tool 무장하기 (Multiple Tools)

## 1. 실험 제목과 목표
- **제목**: 만능 비서로 거듭나기 (Routing)
- **목표**: 교안 슬라이드에 등장한 계산기, 검색기(DuckDuckGo), 사내 문서 검색기(Retriever) 등 3가지 각기 다른 툴을 한꺼번에 LLM에게 쥐어줍니다. 사용자의 질문에 따라 LLM이 스스로 어떤 툴을 써야 할지 정확히 판단(Routing)하는지 테스트합니다.

## 2. 실험 개요
1. **Tool 1 (계산기)**: 파이썬 사칙연산 함수
2. **Tool 2 (웹 검색기)**: LangChain에서 기본 제공하는 DuckDuckGo Search 툴 활용 (무료)
3. **Tool 3 (사내 문서 검색기)**: 04 챕터에서 배웠던 Chroma Vector DB를 툴로 변환 (`create_retriever_tool`)
4. **실험**: 세 가지 성격이 전혀 다른 질문을 연속으로 던져 LLM의 툴 선택 능력 확인

## 3. 라이브러리 import
> **참고**: 인터넷 검색을 위해 `duckduckgo-search` 패키지가 필요합니다. (`pip install duckduckgo-search`)

In [2]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.tools.retriever import create_retriever_tool

## 4. 환경 변수 로드 및 Tool 3대장 준비

In [4]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("=== [Tool 3대장 생성 중...] ===")

# [Tool 1] 계산기
@tool
def calculator(expression: str) -> str:
    """수학 계산을 수행합니다. 파이썬의 eval()을 사용하여 계산 가능한 수식을 문자열로 입력하세요. (예: '1500000 * 0.15')"""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"계산 에러: {e}"

# [Tool 2] 웹 검색기 (DuckDuckGo)
# LangChain이 이미 잘 만들어둔 툴 객체입니다.
search_tool = DuckDuckGoSearchRun(
    name="web_search",
    description="최신 인터넷 정보, 뉴스, 날씨 등을 검색할 때 사용합니다."
)

# [Tool 3] 사내 문서 검색기 (RAG)
# 가짜 사내 규정을 DB에 넣고 Retriever Tool로 변환합니다.
docs = [Document(page_content="[사내복지] 매년 11월 11일은 창립기념일로, 전 직원에게 30만 원 상당의 한우 세트를 지급합니다.")]
vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)
retriever = vectorstore.as_retriever()

retriever_tool = create_retriever_tool(
    retriever,
    name="company_policy_search",
    description="사내 규정, 복지, 휴가, 인사 관련 정보를 검색할 때 사용합니다."
)

# LLM에게 3개의 툴을 모두 주입합니다!
tools = [calculator, search_tool, retriever_tool]
llm_with_tools = llm.bind_tools(tools)
print("-> 3개의 툴 (계산기, 웹 검색, 사내망 검색) 장착 완료!")

=== [Tool 3대장 생성 중...] ===
-> 3개의 툴 (계산기, 웹 검색, 사내망 검색) 장착 완료!


## 5. 실험: LLM의 툴 선택 능력 (Routing) 테스트
이제 3가지 각기 다른 질문을 연속으로 던져보고, LLM이 어떻게 반응하는지 `tool_calls` 안의 `name` 값을 확인해봅시다.

In [5]:
questions = [
    "150만 원에서 15% 세금을 떼면 얼마야?",
    "현재 대한민국의 대통령은 누구야?",
    "우리 회사 창립기념일에 뭐 주는 거 있어?"
]

for q in questions:
    print(f"\n[사용자 질문]: {q}")
    msg = llm_with_tools.invoke(q)
    
    # 툴 호출 여부 검사
    if msg.tool_calls:
        for tool_call in msg.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            print(f"🤖 LLM의 판단 -> '{tool_name}' 툴을 써야겠다! (입력값: {tool_args})")
    else:
        # 툴이 필요 없다고 판단한 경우 (보통 단순 인사말 등)
        print(f"🤖 LLM의 판단 -> 툴이 필요 없네. 직접 대답할게! ({msg.content})")

print("\n-> 결과: LLM이 사용자의 의도를 완벽히 파악하여, 수학은 계산기 툴로, 인터넷 정보는 웹 검색 툴로, 회사 정보는 사내 DB 검색 툴로 라우팅(Routing)하는 데 성공했습니다!")


[사용자 질문]: 150만 원에서 15% 세금을 떼면 얼마야?
🤖 LLM의 판단 -> 'calculator' 툴을 써야겠다! (입력값: {'expression': '1500000 * 0.15'})

[사용자 질문]: 현재 대한민국의 대통령은 누구야?
🤖 LLM의 판단 -> 'web_search' 툴을 써야겠다! (입력값: {'query': 'current president of South Korea'})

[사용자 질문]: 우리 회사 창립기념일에 뭐 주는 거 있어?
🤖 LLM의 판단 -> 'company_policy_search' 툴을 써야겠다! (입력값: {'query': '창립기념일 선물'})

-> 결과: LLM이 사용자의 의도를 완벽히 파악하여, 수학은 계산기 툴로, 인터넷 정보는 웹 검색 툴로, 회사 정보는 사내 DB 검색 툴로 라우팅(Routing)하는 데 성공했습니다!


## 6. 결과 해석

1. **라우팅(Routing)의 마법**: LLM은 수많은 if-else 문을 짤 필요 없이, 각 툴의 `description`만 읽고도 스스로 어떤 툴이 필요한지 찾아가는 놀라운 지능을 보여줍니다.
2. **확장성**: 앞으로 이메일 발송 툴(`send_email`), DB 조회 툴(`query_sql`) 등 수십 개의 툴을 만들어 배열에 넣기만 하면, 모델은 말 그대로 **모든 일을 다 할 수 있는 에이전트(Agent)**가 됩니다.
3. **과제**: 그러나 지금 이 코드는 "어떤 툴을 써야 할지 결정"까지만 하고 끝났습니다. 실제로 이걸 실행해서 최종 답까지 얻어내려면 02번 실험처럼 복잡한 `for` 루프와 메세지 관리를 짜야 합니다.

## 7. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* 여러 개의 툴을 동시에 던져줘도 LLM이 헷갈리지 않고 정확한 툴과 입력값을 매칭함.
* 특히 RAG 시간에 만든 `Retriever`를 툴로 변환해서 주면, 챗봇이 필요할 때만 사내 규정을 뒤져보는 똑똑한 녀석으로 변모함.

### 다음 개선 방향
* LLM이 툴을 선택하고 -> 애플리케이션이 실행하고 -> 다시 LLM이 요약하는 이 반복적이고 지루한 루프(Loop)를 우리가 일일이 코딩하는 건 너무 비효율적임.
* LangChain 생태계의 꽃, **LangGraph** 또는 **AgentExecutor**를 도입하여 이 모든 사이클을 "알아서 무한 반복" 하도록 만드는 진정한 **Agent 구축 단계**로 넘어갈 준비 완료!